# Operações em Grandes Volumes e Comparação de Desempenho

Este notebook executa consultas com agrupamento por `ano_censo`, aumentando gradualmente o volume processado conforme a disponibilidade do parquet consolidado.

Bibliotecas comparadas:

- Pandas
- PyArrow
- Polars, quando disponível no ambiente

Como `ano_censo` é a única variável temporal do conjunto, todas as consultas temporais usam apenas esse campo.

## Configuração

In [1]:
from pathlib import Path
from time import perf_counter

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

try:
  import polars as pl
except ImportError:
  pl = None

try:
  import matplotlib.pyplot as plt
except ImportError:
  plt = None

pd.options.display.max_columns = 80
pd.options.display.float_format = '{:,.2f}'.format

candidatos = [Path('censo-escolar.parquet'), Path('trabalho/censo-escolar.parquet')]
PARQUET_PATH = next((path for path in candidatos if path.exists()), None)
if PARQUET_PATH is None:
  raise FileNotFoundError('Arquivo censo-escolar.parquet não encontrado.')

COLUNAS_CONSULTA = ['ano_censo', 'qt_mat_bas', 'qt_mat_fund', 'qt_mat_med']
BATCH_SIZE = 200_000
pf = pq.ParquetFile(PARQUET_PATH)
TOTAL_LINHAS = pf.metadata.num_rows
VOLUMES = sorted({v for v in [100_000, 500_000, 1_000_000, 2_000_000, TOTAL_LINHAS] if v <= TOTAL_LINHAS})

print(f'Arquivo: {PARQUET_PATH}')
print(f'Linhas disponíveis: {TOTAL_LINHAS:,}')
print(f'Volumes testados: {[f"{v:,}" for v in VOLUMES]}')
print(f'Polars disponível: {pl is not None}')
print(f'Matplotlib disponível: {plt is not None}')

Arquivo: censo-escolar.parquet
Linhas disponíveis: 7,376,443
Volumes testados: ['100,000', '500,000', '1,000,000', '2,000,000', '7,376,443']
Polars disponível: False
Matplotlib disponível: False


## Funções Auxiliares

In [2]:
def ler_arrow_n_linhas(n_linhas):
  partes = []
  lidas = 0

  for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=COLUNAS_CONSULTA):
    restante = n_linhas - lidas
    if restante <= 0:
      break

    if batch.num_rows > restante:
      batch = batch.slice(0, restante)

    partes.append(pa.Table.from_batches([batch]))
    lidas += batch.num_rows

  return pa.concat_tables(partes) if partes else pa.table({col: [] for col in COLUNAS_CONSULTA})


def consulta_pandas(n_linhas):
  tabela = ler_arrow_n_linhas(n_linhas)
  df = tabela.to_pandas()
  return (
    df.groupby('ano_censo', as_index=False)
    .agg(
      linhas=('ano_censo', 'size'),
      matriculas_basicas=('qt_mat_bas', 'sum'),
      matriculas_fundamental=('qt_mat_fund', 'sum'),
      matriculas_medio=('qt_mat_med', 'sum'),
    )
    .sort_values('ano_censo')
  )


def consulta_pyarrow(n_linhas):
  tabela = ler_arrow_n_linhas(n_linhas)
  agrupado = tabela.group_by('ano_censo').aggregate([
    ('ano_censo', 'count'),
    ('qt_mat_bas', 'sum'),
    ('qt_mat_fund', 'sum'),
    ('qt_mat_med', 'sum'),
  ])
  return (
    agrupado.to_pandas()
    .rename(columns={
      'ano_censo_count': 'linhas',
      'qt_mat_bas_sum': 'matriculas_basicas',
      'qt_mat_fund_sum': 'matriculas_fundamental',
      'qt_mat_med_sum': 'matriculas_medio',
    })
    .sort_values('ano_censo')
    .reset_index(drop=True)
  )


def consulta_polars(n_linhas):
  if pl is None:
    return None

  consulta = pl.scan_parquet(str(PARQUET_PATH)).select(COLUNAS_CONSULTA)
  if n_linhas < TOTAL_LINHAS:
    consulta = consulta.head(n_linhas)

  return (
    consulta.group_by('ano_censo')
    .agg([
      pl.len().alias('linhas'),
      pl.col('qt_mat_bas').sum().alias('matriculas_basicas'),
      pl.col('qt_mat_fund').sum().alias('matriculas_fundamental'),
      pl.col('qt_mat_med').sum().alias('matriculas_medio'),
    ])
    .sort('ano_censo')
    .collect()
    .to_pandas()
  )

## Consultas Com Incremento Gradual de Volume

In [3]:
consultas = [
  ('Pandas', consulta_pandas),
  ('PyArrow', consulta_pyarrow),
]
if pl is not None:
  consultas.append(('Polars', consulta_polars))

resultados = []
tempos = []

for volume in VOLUMES:
  for biblioteca, funcao in consultas:
    inicio = perf_counter()
    resultado = funcao(volume)
    segundos = perf_counter() - inicio

    tempos.append({
      'biblioteca': biblioteca,
      'linhas_processadas': volume,
      'segundos': segundos,
    })

    resultado = resultado.copy()
    resultado.insert(0, 'biblioteca', biblioteca)
    resultado.insert(1, 'linhas_processadas', volume)
    resultados.append(resultado)
    print(f'{biblioteca:7} | {volume:>10,} linhas | {segundos:>8.3f}s')

benchmark = pd.DataFrame(tempos)
agrupamentos = pd.concat(resultados, ignore_index=True)
display(benchmark)

Pandas  |    100,000 linhas |    0.025s
PyArrow |    100,000 linhas |    0.012s
Pandas  |    500,000 linhas |    0.036s
PyArrow |    500,000 linhas |    0.008s
Pandas  |  1,000,000 linhas |    0.055s
PyArrow |  1,000,000 linhas |    0.012s
Pandas  |  2,000,000 linhas |    0.095s
PyArrow |  2,000,000 linhas |    0.012s
Pandas  |  7,376,443 linhas |    0.612s
PyArrow |  7,376,443 linhas |    0.091s


,biblioteca,linhas_processadas,segundos
0,Pandas,100000,0.02
1,PyArrow,100000,0.01
2,Pandas,500000,0.04
3,PyArrow,500000,0.01
4,Pandas,1000000,0.06
5,PyArrow,1000000,0.01
6,Pandas,2000000,0.09
7,PyArrow,2000000,0.01
8,Pandas,7376443,0.61
9,PyArrow,7376443,0.09


## Resultado da Consulta por Ano

In [4]:
maior_volume = max(VOLUMES)
consulta_final = agrupamentos[
  (agrupamentos['linhas_processadas'] == maior_volume)
  & (agrupamentos['biblioteca'] == consultas[0][0])
].drop(columns=['biblioteca', 'linhas_processadas'])

display(consulta_final)

,ano_censo,linhas,matriculas_basicas,matriculas_fundamental,matriculas_medio
30,1995,243637,0.00,0.00,0.00
31,1996,276731,0.00,0.00,0.00
32,1997,273951,0.00,0.00,0.00
33,1998,267532,0.00,0.00,0.00
34,1999,266645,0.00,0.00,0.00
35,2000,261988,0.00,0.00,0.00
36,2001,264735,0.00,0.00,0.00
37,2002,256986,0.00,0.00,0.00
38,2003,253405,0.00,0.00,0.00
39,2004,248257,0.00,0.00,0.00


## Gráficos

In [5]:
if plt is None:
  print('Matplotlib não está disponível neste ambiente. As tabelas acima substituem os gráficos.')
else:
  fig, ax = plt.subplots(figsize=(10, 5))
  for biblioteca, dados in benchmark.groupby('biblioteca'):
    ax.plot(dados['linhas_processadas'], dados['segundos'], marker='o', label=biblioteca)
  ax.set_title('Tempo de agrupamento por volume processado')
  ax.set_xlabel('Linhas processadas')
  ax.set_ylabel('Segundos')
  ax.legend()
  ax.grid(True, alpha=0.3)
  plt.show()

  consulta_final.plot(
    x='ano_censo',
    y=['matriculas_basicas', 'matriculas_fundamental', 'matriculas_medio'],
    figsize=(11, 5),
    marker='o',
    title='Matrículas por ano do Censo Escolar',
  )
  plt.xlabel('Ano do censo')
  plt.ylabel('Matrículas')
  plt.grid(True, alpha=0.3)
  plt.show()

Matplotlib não está disponível neste ambiente. As tabelas acima substituem os gráficos.


## Comparação de Desempenho

In [6]:
comparacao = (
  benchmark.pivot(index='linhas_processadas', columns='biblioteca', values='segundos')
  .reset_index()
)

if {'Pandas', 'PyArrow'}.issubset(comparacao.columns):
  comparacao['speedup_pyarrow_vs_pandas'] = comparacao['Pandas'] / comparacao['PyArrow']
if {'Pandas', 'Polars'}.issubset(comparacao.columns):
  comparacao['speedup_polars_vs_pandas'] = comparacao['Pandas'] / comparacao['Polars']

display(comparacao)

biblioteca,linhas_processadas,Pandas,PyArrow,speedup_pyarrow_vs_pandas
0,100000,0.02,0.01,2.01
1,500000,0.04,0.01,4.60
2,1000000,0.06,0.01,4.76
3,2000000,0.09,0.01,8.03
4,7376443,0.61,0.09,6.68


## Observações

- A comparação usa a mesma consulta lógica nas três bibliotecas: agrupar por `ano_censo` e somar matrículas.
- Pandas recebe os dados convertidos de Arrow para DataFrame antes do agrupamento.
- PyArrow agrupa diretamente em tabela Arrow.
- Polars usa execução lazy quando a biblioteca está instalada.
- Os tempos podem variar entre execuções por cache de disco, carga do sistema e custo de conversão entre formatos.